# NER LLM Prompts
This pipeline uses an LLM (either local or from HF) to perform NER using system prompts

In [ ]:
%load_ext autoreload 
%autoreload 2

import os
from dotenv import load_dotenv
from models import init_llm
from prompts import guidelines_llama3, guidelines_gemma3, guidelines_llama3_legal, guidelines_llama3_legal_lite

load_dotenv()

token = os.environ["HF_TOKEN"]
# model_name = "google/gemma-3-4b-it"
# model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
model_name = "meta-llama/Llama-3.1-8B-Instruct"
# model_name = "mistralai/Ministral-8B-Instruct-2410"

prompt = guidelines_llama3_legal_lite

model, tokenizer = init_llm(
    model_name,
    token,
    do_quantization=True,
)

# Necessary for Llama3.1's prompt
# tokenizer.chat_template = tokenizer.chat_template.replace('{{- "Cutting Knowledge Date: December 2023\\n" }}\n{{- "Today Date: " + date_string + "\\n\\n" }}\n', "")

In [ ]:
from models import  create_query, query_llm


input_text = """Il Riccardo Galli ha specificato di aver conosciuto il Paride Valli presso la macelleria, sita nel quartiere Salamini, dei propri cugini Ettore Vitale, Ottavio Neri ed Leandro Sala (l’altro collaborante già citato), frequentata dal padre Riccardo Galli e da altri uomini d’altri tempi (come Gambino Simone Costa) e che il Paride Valli era persona vicina ed amica di Tullio Conte Di Napoli, circostanza anche personalmente constatata dal collaboratore allorché, in alcune occasioni, aveva incontrato il Paride Valli in compagnia del Di Napoli nella villa di proprietà di Enrico Fontana Gino Basile, altro uomo d’altri tempi della famiglia di Orlando Pace, nonché parente dello stesso Paride Valli ove lo stesso Di Napoli dimorava durante la sua gita."""

# query = tokenizer.apply_chat_template(
#     guidelines_messages + [{"role": "user", "content": input_text}],
#     tokenize=False,
# )

query = create_query(prompt, input_text)

response = query_llm(model, tokenizer, query)

response.split("Output: \n")[-1]

In [ ]:
import json
from sklearn.model_selection import train_test_split

from models import extract_ner

# Load jsonl file into a list of dictionaries
with open("../data/labelled/all.jsonl") as test_f:
    data = [json.loads(line) for line in test_f]

# Create test split (notice: fixed seed)
_, test_data = train_test_split(data, test_size=0.3, random_state=52)

# Extract NER tags from input sentences
pred = extract_ner(model, tokenizer, prompt, test_data)

# Attach NER tags to original dataset
for sent, pred in zip(test_data, pred):
    sent["pred"] = pred

In [ ]:
from datetime import datetime

# Create timestamp to uniquely identify output files
dt = str(datetime.now()).split(".")[0].replace(" ", "_").replace(":", "-")

# Save predictions to file
model_name = model_name.replace("../", "")
output_file = f"../output/{dt}_llm_ner_pred_{model_name.split('/')[-1]}.txt"
with open(output_file, "w") as test_f:
    json.dump(test_data, test_f)

## Preprocessing

In [ ]:
%load_ext autoreload 
%autoreload 2

In [ ]:
import json

# example
# output_file = "../output/llm_ner_pred_Llama-3.1-8B-Instruct_2025-08-20_18-58-19.txt"
# output_file = "../output/2025-08-26_09-18-52_llm_ner_pred_Llama-3.1-8B-Instruct.txt"


with open(output_file, "r") as f:
    # output = [line.strip() for line in f.readlines()]
    raw_dataset = json.load(f)

In [ ]:
# Additionally cleaning needed for gemma
if "gemma" in model_name:
    for item in raw_dataset:
        item["pred"] = item["pred"].split("Output:\n")[-1]

In [ ]:
# For each line in output, extract the predicted entity, its tag and its start/end indexes using regular expressions
from util import process_raw_dataset

dataset = process_raw_dataset(raw_dataset)

# Save as jsonl file
with open(output_file.replace(".txt", "_eval.json"), "w") as file:
    file.write(json.dumps(dataset, ensure_ascii=False))

print(f"Saved predictions to {output_file}")

## Evaluation

In [ ]:
texts = [el["text"] for el in dataset]
texts_pred = [el["text_pred"] for el in dataset]
y_true = [el["label"] for el in dataset]
y_pred = [el["label_pred"] for el in dataset]

In [ ]:
# Entity-only evaluation
from itertools import chain
from util import evaluate_ner

# Flatten lists to evaluate metrics on whole dataset
flat_preds = list(map(tuple, chain.from_iterable(y_pred)))
flat_golds = list(map(tuple, chain.from_iterable(y_true)))

# Normalize style to ensure fair comparisons
flat_preds = [(w[0].strip().title(), w[1]) for w in flat_preds]
flat_golds = [(w[0].strip().title(), w[1]) for w in flat_golds]

results = evaluate_ner(flat_preds, flat_golds)
print("NER evaluation results")
results

In [ ]:
# Entity-only evaluation
from itertools import chain
from util import evaluate_ner

# Flatten lists to evaluate metrics on whole dataset
flat_preds = list(map(tuple, chain.from_iterable(y_pred)))
flat_golds = list(map(tuple, chain.from_iterable(y_true)))

# Normalize style to ensure fair comparisons
flat_preds = [w[0].strip().title() for w in flat_preds]
flat_golds = [w[0].strip().title() for w in flat_golds]

tokenlevel_results = evaluate_ner(flat_preds, flat_golds)
print("Token-level results (token only, no label)")
tokenlevel_results

In [ ]:
file_path = "../experiments_results.json"

# Load existing results
with open(file_path, "r") as f:
    results_dict = json.load(f)

# Add new result under filename
clean_name = output_file.split("/")[-1].replace(".txt", "")
if clean_name not in results_dict:
    results_dict[clean_name] = results

    # Save back
    with open(file_path, "w") as f:
        json.dump(results_dict, f, indent=4)

## Analysis

In [ ]:
import pandas as pd

# Flatten lists to evaluate metrics on whole dataset
pred_col, true_col, prec_col, rec_col, f1_col = [], [], [], [], []
for pred, t in zip(y_pred, y_true):
    # flat_preds = list(map(tuple, chain.from_iterable(p)))
    # flat_golds = list(map(tuple, chain.from_iterable(t)))
    if not pred or not t:
        continue

    pred = [(w[0].title().strip(), w[1]) for w in pred]
    t = [(w[0].title().strip(), w[1]) for w in t]

    res = evaluate_ner(pred, t)
    pred_col.append(pred)
    true_col.append(t)
    prec_col.append(res["precision"])
    rec_col.append(res["recall"])
    f1_col.append(res["f1"])


df = pd.DataFrame([texts, texts_pred, pred_col, true_col, prec_col, rec_col, f1_col]).T
df.columns = ["text", "texts_pred", "pred", "true", "precision", "recall", "f1"]

In [ ]:
print(df["precision"].mean())
print(df["recall"].mean())
print(df["f1"].mean())

In [ ]:
tmp_df = df.sort_values(by="precision")

idx = 15
print(tmp_df.iloc[idx]["text"])
print(tmp_df.iloc[idx]["texts_pred"])
print()
print("pred: ", tmp_df.iloc[idx]["pred"])
print("true: ", tmp_df.iloc[idx]["true"])
print()
print("true: ", tmp_df.iloc[idx][["precision", "recall", "f1"]])